
##Import the required library

In [0]:
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.types import * 



##Create Catalog,Schema and Volume

In [0]:
spark.sql('CREATE CATALOG IF NOT EXISTS cricket_api')
spark.sql('USE CATALOG cricket_api')
spark.sql('CREATE SCHEMA IF NOT EXISTS cricket_api.cricket')
spark.sql('CREATE VOLUME IF NOT EXISTS cricket_api.cricket.cricket_api_project')


In [0]:
base_path = '/Volumes/cricket_api/cricket/cricket_api_project'


###Calling Cricket API

In [0]:
API_KEY ='74f6fed4-72fe-4e8c-8ea3-f11dcde36615'

api_url = f'https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0'

response = requests.get(api_url)
response.raise_for_status()

api_data = response.json()
print(api_data.keys())

print(json.dumps(api_data,indent=4))

###Save Raw API Response in Volume

In [0]:
raw_file_path = f'{base_path}/raw_current_matches.json'

with open (raw_file_path, 'w') as file:
    json.dump(api_data,file)

print(f'RAW file saved at : {raw_file_path}')


###Create Bronze Layer DataFrame or Table 

In [0]:
bronze_data = [{
    "source_api" : api_url,
    "raw_json" : json.dumps(api_data),
    "ingestion_time" : None

}]

bronze_schema = StructType([
    StructField("source_api", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_time", TimestampType(),True)
])

In [0]:
bronze_schema

In [0]:
bronze_df = spark.createDataFrame(bronze_data,schema=bronze_schema)\
    .withColumn('ingestion_time',current_timestamp())

bronze_df.display()


###Save the bronze table

In [0]:
bronze_df.write.format('delta')\
    .mode('overwrite')\
        .saveAsTable('cricket_api.cricket.bronze_current_matches')

In [0]:
%sql

Select * from cricket_api.cricket.bronze_current_matches